# NumPyro Mixture Model With Collapsed NUTS

This notebook keeps the same generative logic as `mcmc_mixture.ipynb`, but it changes the inference strategy:

- The discrete assignment `z` is integrated out during inference.
- The gated latent variables `x_1`, `x_2`, and selected `x` are also integrated out of the likelihood.
- Inference uses plain `NUTS` on the collapsed model.
- After fitting, posterior predictive sampling reconstructs `z`, `x_1`, `x_2`, `x`, and replicated observations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import numpyro

NUM_WANTED_CHAINS = 4
numpyro.set_host_device_count(NUM_WANTED_CHAINS)

import jax
import jax.numpy as jnp
from jax import random

import numpyro.distributions as dist
from numpyro import handlers
from numpyro.infer import MCMC, NUTS, Predictive

In [ ]:
N = 1000
NUM_WARMUP = 1000
NUM_SAMPLES = 1000
NUM_CHAINS = NUM_WANTED_CHAINS
NUM_PPC_SAMPLES = 400

SIGMA_X1 = 1.0
SIGMA_X2 = 1.5
SIGMA_OBS = 0.25

rng_key = random.PRNGKey(0)
print("jax.local_device_count()", jax.local_device_count())

In [ ]:
def explicit_mixture_model(obs=None, N=1000):
    K = 2

    w = numpyro.sample("w", dist.Dirichlet(jnp.ones(K)))
    theta_1 = numpyro.sample("theta_1", dist.Uniform(-1.0, 0.0))
    theta_2 = numpyro.sample("theta_2", dist.Uniform(2.0, 3.0))

    with numpyro.plate("n", N):
        z = numpyro.sample("z", dist.Categorical(probs=w))
        x_1 = numpyro.sample("x_1", dist.Normal(theta_1, SIGMA_X1))
        x_2 = numpyro.sample("x_2", dist.Normal(theta_2, SIGMA_X2))

        # K-way friendly gate: gather selected component per data point
        x_components = jnp.stack([x_1, x_2], axis=-1)      # shape (N, K)
        x = jnp.take_along_axis(x_components, z[:, None], axis=1).squeeze(-1)

        numpyro.deterministic("x", x)
        numpyro.sample("obs", dist.Normal(x, SIGMA_OBS), obs=obs)

In [ ]:
data_key, mcmc_key, predictive_key = random.split(rng_key, 3)
prior_trace = handlers.trace(handlers.seed(explicit_mixture_model, data_key)).get_trace(N=N)

init_state = {
    name: site["value"]
    for name, site in prior_trace.items()
    if site["type"] in {"sample", "deterministic"}
}
observed = init_state["obs"]

print("w", init_state["w"])
print("theta_1", init_state["theta_1"])
print("theta_2", init_state["theta_2"])
print("obs shape", observed.shape)

In [ ]:
obs_np = np.asarray(observed)
z_np = np.asarray(init_state["z"])

theta_1_true = float(init_state["theta_1"])
theta_2_true = float(init_state["theta_2"])
w_true = np.asarray(init_state["w"])

plt.figure(figsize=(10, 5))
plt.hist(obs_np, bins=30, density=True, histtype="step", label="Observed")
plt.hist(obs_np[z_np == 0], bins=30, density=True, histtype="step", linestyle="--", color="red", label="Component 1")
plt.hist(obs_np[z_np == 1], bins=30, density=True, histtype="step", linestyle="--", color="blue", label="Component 2")

xx = np.linspace(-5.0, 10.0, 300)
pdf_0 = w_true[0] * np.exp(np.asarray(dist.Normal(theta_1_true, SIGMA_X1).log_prob(jnp.asarray(xx))))
pdf_1 = w_true[1] * np.exp(np.asarray(dist.Normal(theta_2_true, SIGMA_X2).log_prob(jnp.asarray(xx))))
plt.plot(xx, pdf_0, "r--", label="Component 1 PDF")
plt.plot(xx, pdf_1, "b--", label="Component 2 PDF")
plt.plot(xx, pdf_0 + pdf_1, "k--", label="Mixture PDF")
plt.axvline(theta_1_true, color="red", linestyle=":", label="True theta_1")
plt.axvline(theta_2_true, color="blue", linestyle=":", label="True theta_2")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
def collapsed_mixture_model(obs=None, N=1000):
    K = 2

    w = numpyro.sample("w", dist.Dirichlet(jnp.ones(K)))
    theta_1 = numpyro.sample("theta_1", dist.Uniform(-1.0, 0.0))
    theta_2 = numpyro.sample("theta_2", dist.Uniform(2.0, 3.0))

    component_loc = jnp.stack([theta_1, theta_2])
    component_scale = jnp.array([
        jnp.sqrt(SIGMA_X1 ** 2 + SIGMA_OBS ** 2),
        jnp.sqrt(SIGMA_X2 ** 2 + SIGMA_OBS ** 2),
    ])

    with numpyro.plate("n", N):
        numpyro.sample(
            "obs",
            dist.MixtureSameFamily(
                dist.Categorical(probs=w),
                dist.Normal(component_loc, component_scale),
            ),
            obs=obs,
        )

In [ ]:
kernel = NUTS(collapsed_mixture_model)
mcmc = MCMC(
    kernel,
    num_warmup=NUM_WARMUP,
    num_samples=NUM_SAMPLES,
    num_chains=NUM_CHAINS,
    progress_bar=True,
)

mcmc.run(mcmc_key, obs=observed, N=N)
mcmc.print_summary()

In [ ]:
posterior_chain = mcmc.get_samples(group_by_chain=True)
posterior = {name: np.asarray(value) for name, value in posterior_chain.items()}
posterior_all = mcmc.get_samples()

print("theta_1 true", theta_1_true)
print("theta_1 mean", posterior["theta_1"].mean())
print("theta_2 true", theta_2_true)
print("theta_2 mean", posterior["theta_2"].mean())
print("w true", w_true)
print("w mean", posterior["w"].mean(axis=(0, 1)))

plt.figure(figsize=(10, 4))
for chain in range(posterior["theta_1"].shape[0]):
    plt.plot(posterior["theta_1"][chain], alpha=0.5)
plt.axhline(theta_1_true, color="red", linestyle="--", label="True theta_1")
plt.title("Trace plot for theta_1")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
for chain in range(posterior["theta_2"].shape[0]):
    plt.plot(posterior["theta_2"][chain], alpha=0.5)
plt.axhline(theta_2_true, color="blue", linestyle="--", label="True theta_2")
plt.title("Trace plot for theta_2")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
def latent_posterior_predictive_model(obs, N=1000):
    K = 2

    w = numpyro.sample("w", dist.Dirichlet(jnp.ones(K)))
    theta_1 = numpyro.sample("theta_1", dist.Uniform(-1.0, 0.0))
    theta_2 = numpyro.sample("theta_2", dist.Uniform(2.0, 3.0))

    collapsed_scale_1 = jnp.sqrt(SIGMA_X1 ** 2 + SIGMA_OBS ** 2)
    collapsed_scale_2 = jnp.sqrt(SIGMA_X2 ** 2 + SIGMA_OBS ** 2)

    post_var_1 = 1.0 / (1.0 / SIGMA_X1 ** 2 + 1.0 / SIGMA_OBS ** 2)
    post_var_2 = 1.0 / (1.0 / SIGMA_X2 ** 2 + 1.0 / SIGMA_OBS ** 2)
    post_scale_1 = jnp.sqrt(post_var_1)
    post_scale_2 = jnp.sqrt(post_var_2)

    with numpyro.plate("n", N):
        logits = jnp.stack([
            jnp.log(w[0]) + dist.Normal(theta_1, collapsed_scale_1).log_prob(obs),
            jnp.log(w[1]) + dist.Normal(theta_2, collapsed_scale_2).log_prob(obs),
        ], axis=-1)

        z = numpyro.sample("z", dist.Categorical(logits=logits))

        x_1_loc_active = post_var_1 * (theta_1 / SIGMA_X1 ** 2 + obs / SIGMA_OBS ** 2)
        x_2_loc_active = post_var_2 * (theta_2 / SIGMA_X2 ** 2 + obs / SIGMA_OBS ** 2)

        x_1 = numpyro.sample(
            "x_1",
            dist.Normal(
                jnp.where(z == 0, x_1_loc_active, theta_1),
                jnp.where(z == 0, post_scale_1, SIGMA_X1),
            ),
        )
        x_2 = numpyro.sample(
            "x_2",
            dist.Normal(
                jnp.where(z == 1, x_2_loc_active, theta_2),
                jnp.where(z == 1, post_scale_2, SIGMA_X2),
            ),
        )

        x = jnp.where(z == 0, x_1, x_2)
        numpyro.deterministic("x", x)
        numpyro.sample("obs_rep", dist.Normal(x, SIGMA_OBS))

In [ ]:
num_available = posterior_all["w"].shape[0]
num_ppc = min(NUM_PPC_SAMPLES, num_available)
ppc_index = np.linspace(0, num_available - 1, num_ppc, dtype=int)
posterior_subset = {name: value[ppc_index] for name, value in posterior_all.items()}

predictive = Predictive(
    latent_posterior_predictive_model,
    posterior_samples=posterior_subset,
    return_sites=["z", "x_1", "x_2", "x", "obs_rep"],
)
ppc = predictive(predictive_key, obs=observed, N=N)
ppc_np = {name: np.asarray(value) for name, value in ppc.items()}

print("z samples", ppc_np["z"].shape)
print("x samples", ppc_np["x"].shape)
print("obs_rep samples", ppc_np["obs_rep"].shape)

In [ ]:
theta_1_fit = posterior["theta_1"].mean()
theta_2_fit = posterior["theta_2"].mean()
w_fit = posterior["w"].mean(axis=(0, 1))

plt.figure(figsize=(10, 5))
plt.hist(obs_np, bins=30, density=True, histtype="step", label="Observed")

pdf_0_fit = w_fit[0] * np.exp(np.asarray(dist.Normal(theta_1_fit, SIGMA_X1).log_prob(jnp.asarray(xx))))
pdf_1_fit = w_fit[1] * np.exp(np.asarray(dist.Normal(theta_2_fit, SIGMA_X2).log_prob(jnp.asarray(xx))))
plt.plot(xx, pdf_0_fit, "r-", label="Fitted component 1 PDF")
plt.plot(xx, pdf_1_fit, "b-", label="Fitted component 2 PDF")
plt.plot(xx, pdf_0_fit + pdf_1_fit, "k-", label="Fitted mixture PDF")
plt.axvline(theta_1_true, color="red", linestyle="--", label="True theta_1")
plt.axvline(theta_2_true, color="blue", linestyle="--", label="True theta_2")
plt.axvline(theta_1_fit, color="red", linestyle=":", label="Posterior mean theta_1")
plt.axvline(theta_2_fit, color="blue", linestyle=":", label="Posterior mean theta_2")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

p_component_0 = (ppc_np["z"] == 0).mean(axis=0)
p_component_1 = (ppc_np["z"] == 1).mean(axis=0)

plt.figure(figsize=(10, 5))
plt.scatter(obs_np, p_component_0, s=6, alpha=0.5, label="P(z = 0 | obs)")
plt.scatter(obs_np, p_component_1, s=6, alpha=0.5, label="P(z = 1 | obs)")
plt.xlabel("Observed value")
plt.ylabel("Posterior assignment probability")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(ppc_np["obs_rep"].reshape(-1), bins=60, density=True, histtype="step", label="Posterior predictive obs")
plt.hist(obs_np, bins=30, density=True, histtype="step", label="Observed")
plt.grid(alpha=0.3)
plt.legend()
plt.show()